In [ ]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

# 1. Environment and Database Connection
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# 2. Fetch the engineered data
df = pd.read_sql("SELECT * FROM engineered_housing_data", engine)

# --- 01-DƏN GƏLƏN VACİB ƏLAVƏ: TARGET LOG TRANSFORMATION ---
# Qiymət çox böyük rəqəm olduğu üçün onun log-unu götürmək modelin dəqiqliyini artırır.
# Diqqət: Modeldən proqnoz alanda sonda 'exp' edib geri qaytaracağıq.
df['sale_price'] = np.log1p(df['sale_price'])

# 3. Features and Target
X = df.drop('sale_price', axis=1)
y = df['sale_price']

# 4. Split: 80% Training, 20% Final Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Persist splits to SQL 
# y_train və y_test artıq log-lanmış qiymətlərdir.
X_train.to_sql('X_train', engine, if_exists='replace', index=False)
X_test.to_sql('X_test', engine, if_exists='replace', index=False)
y_train.to_frame().to_sql('y_train', engine, if_exists='replace', index=False)
y_test.to_frame().to_sql('y_test', engine, if_exists='replace', index=False)

print("Step 04: Train-Test split (with Logged Target) successfully saved to SQL.")